<a href="https://colab.research.google.com/github/NguyenHuy2003/7080325_flutter/blob/main/train_model_xq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

huyzakaito_mura_clahe_320_path = kagglehub.dataset_download('huyzakaito/mura-clahe-320')

print('Data source import complete.')


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications, optimizers, callbacks
import numpy as np
import os
import glob

# ==============================================================================
# 1. THIẾT LẬP CHIẾN LƯỢC ĐA GPU VÀ TỰ ĐỘNG TÌM ĐƯỜNG DẪN
# ==============================================================================
policy = tf.keras.mixed_precision.Policy('mixed_float16')
tf.keras.mixed_precision.set_global_policy(policy)

strategy = tf.distribute.MirroredStrategy()
print(f"Số lượng GPU đang sử dụng: {strategy.num_replicas_in_sync}")

# --- ĐOẠN CODE TỰ ĐỘNG TÌM ĐƯỜNG DẪN DỮ LIỆU ---
import os

TRAIN_DIR = ""
VAL_DIR = ""

print("Đang tự động dò tìm vị trí ảnh trên Kaggle...")
for root, dirs, files in os.walk('/kaggle/input'):
    if '/train/' in root.replace('\\', '/') and any(f.endswith('.png') for f in files):
        TRAIN_DIR = root.replace('\\', '/').split('/train/')[0] + '/train/'
    if '/valid/' in root.replace('\\', '/') and any(f.endswith('.png') for f in files):
        VAL_DIR = root.replace('\\', '/').split('/valid/')[0] + '/valid/'

    if TRAIN_DIR and VAL_DIR:
        break # Tìm thấy cả 2 thì dừng tìm kiếm cho nhanh

print(f"TRAIN_DIR: {TRAIN_DIR}")
print(f"VAL_DIR: {VAL_DIR}")
# ----------------------------------------------

# Kaggle chỉ cho phép lưu file vào thư mục /kaggle/working/
CHECKPOINT_DIR = '/kaggle/working/'

IMG_SIZE = 320
GLOBAL_BATCH_SIZE = 32 * strategy.num_replicas_in_sync
AUTOTUNE = tf.data.AUTOTUNE

PARTS_MAP = {
    'XR_ELBOW': 0, 'XR_FINGER': 1, 'XR_FOREARM': 2,
    'XR_HAND': 3, 'XR_HUMERUS': 4, 'XR_SHOULDER': 5, 'XR_WRIST': 6
}

# 2. Data pipeline
def load_multitask_data(data_dir):
    print(f"Đọc thư mục: {data_dir}")
    image_paths = glob.glob(os.path.join(data_dir, "**", "*.png"), recursive=True)
    abnormal_labels, part_labels = [], []

    for path in image_paths:
        norm_path = path.replace('\\', '/')
        abnormal_labels.append(1 if 'positive' in norm_path else 0)

        part_found = False
        for part_name, part_idx in PARTS_MAP.items():
            if part_name in norm_path:
                part_labels.append(part_idx)
                part_found = True
                break
        if not part_found: part_labels.append(-1)

    print(f"   -> Đã load {len(image_paths)} ảnh.")
    return image_paths, abnormal_labels, part_labels

def process_path_mtl(file_path, ab_label, part_label):
    img = tf.io.read_file(file_path)
    img = tf.io.decode_png(img, channels=1)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img.set_shape([IMG_SIZE, IMG_SIZE, 1])

    labels_dict = {'abnormality_out': ab_label, 'part_out': part_label}
    return img, labels_dict

def create_mtl_dataset(paths, ab_labels, part_labels, is_training=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, ab_labels, part_labels))
    ds = ds.map(process_path_mtl, num_parallel_calls=AUTOTUNE)
    if is_training:
        ds = ds.shuffle(buffer_size=10000)
    return ds.batch(GLOBAL_BATCH_SIZE).prefetch(AUTOTUNE)

train_paths, train_ab_labels, train_part_labels = load_multitask_data(TRAIN_DIR)
val_paths, val_ab_labels, val_part_labels = load_multitask_data(VAL_DIR)

train_ds = create_mtl_dataset(train_paths, train_ab_labels, train_part_labels, is_training=True)
val_ds = create_mtl_dataset(val_paths, val_ab_labels, val_part_labels, is_training=False)

# 3. Xây dựng mô hình và compole bên trong strategy scope
def build_mtl_model(input_shape=(320, 320, 1)):
    inputs = layers.Input(shape=input_shape)

    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.05)(x)
    x = layers.RandomZoom(0.05)(x)

    x = layers.Conv2D(3, (3, 3), padding='same', use_bias=False)(x)

    backbone = applications.DenseNet169(include_top=False, weights='imagenet', input_shape=(320, 320, 3))
    backbone.trainable = False
    shared_features = backbone(x, training=False)

    # Nhánh 1: Dự đoán bệnh (Main Task)
    avg_pool_1 = layers.GlobalAveragePooling2D()(shared_features)
    max_pool_1 = layers.GlobalMaxPooling2D()(shared_features)
    x1 = layers.Concatenate()([avg_pool_1, max_pool_1])

    x1 = layers.BatchNormalization()(x1)
    x1 = layers.Dropout(0.3)(x1)
    x1 = layers.Dense(512, activation='relu')(x1)
    x1 = layers.BatchNormalization()(x1)
    x1 = layers.Dropout(0.3)(x1)
    # Lưu ý: Ép dtype='float32' cho lớp cuối cùng khi dùng Mixed Precision
    out_abnormality = layers.Dense(1, activation='sigmoid', name='abnormality_out', dtype='float32')(x1)

    # Nhánh 2: Phân loại xương (Auxiliary Task)
    x2 = layers.GlobalAveragePooling2D()(shared_features)
    x2 = layers.Dense(256, activation='relu')(x2)
    x2 = layers.Dropout(0.2)(x2)
    out_part = layers.Dense(7, activation='softmax', name='part_out', dtype='float32')(x2)

    return models.Model(inputs=inputs, outputs=[out_abnormality, out_part], name="MURA_MTL_DualGPU")

# Mọi thứ liên quan đến tạo model, hàm loss, và compile ĐỀU PHẢI NẰM TRONG SCOPE
with strategy.scope():
    model = build_mtl_model()

    # Khởi tạo Focal Loss
    focal_loss = tf.keras.losses.BinaryFocalCrossentropy(gamma=2.0)

    LOSSES = {
        'abnormality_out': focal_loss,
        'part_out': 'sparse_categorical_crossentropy'
    }

    LOSS_WEIGHTS = {'abnormality_out': 1.0, 'part_out': 0.2}

    METRICS = {
        'abnormality_out': [tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.Recall(name='recall')],
        'part_out': ['accuracy']
    }

    # Compile Giai đoạn 1
    model.compile(optimizer=optimizers.Adam(learning_rate=1e-3),
                  loss=LOSSES, loss_weights=LOSS_WEIGHTS, metrics=METRICS)

model.summary()

# 4. Callback và huấn luyện
checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_mura_multitask.keras')

my_callbacks = [
    callbacks.ModelCheckpoint(checkpoint_path, monitor='val_abnormality_out_auc', verbose=1, save_best_only=True, mode='max'),
    callbacks.EarlyStopping(monitor='val_abnormality_out_auc', patience=6, restore_best_weights=True, mode='max'),
    callbacks.ReduceLROnPlateau(monitor='val_abnormality_out_auc', factor=0.2, patience=2, min_lr=1e-6, verbose=1),
    callbacks.CSVLogger(os.path.join(CHECKPOINT_DIR, 'training_log.csv'), append=True)
]

print("\nGiai đoạn 1: Transfer Learning")
history_1 = model.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=my_callbacks)

print("\nGiai đoạn 2:Fine-Tuning Toàn bộ mạng")
# Chuyển đổi trạng thái trainable phải thực hiện lại compile bên trong scope
with strategy.scope():
    model.trainable = True
    model.compile(optimizer=optimizers.Adam(learning_rate=1e-5),
                  loss=LOSSES, loss_weights=LOSS_WEIGHTS, metrics=METRICS)

history_fine = model.fit(
    train_ds, validation_data=val_ds, epochs=20,
    initial_epoch=history_1.epoch[-1],
    callbacks=my_callbacks
)

print(f"\nHoàn tất! Model đã được lưu tại: {checkpoint_path}")